# 13. 글자 단위 사전학습 인코더 (KoCharELECTRA) 로 문맥 판단
12b (외부 모델 없음, 검증 0.9871) 의 점수에 **KoCharELECTRA** (음절 단위 한국어 ELECTRA, Apache-2.0) 두 항을 더한다. LLM 은 아니지만 **외부 사전학습 모델**을 쓰는 단계다.

| 구분 | 단계 | 검증 F1 |
|---|---|---|
| 외부 모델 없음 | 01 | 0.9834 |
| 외부 모델 없음 | 12b | 0.9871 |
| 사전학습 인코더 | **13** | ? |
| LLM (Qwen 1.5B) | 09 (같은 후보) | 0.9908 |
| LLM (Qwen 1.5B) | 11b | 0.9915 |

더하는 두 항:
- **generator (19M)**: 후보 어절의 글자를 하나씩 가리고 맞히는 확률. train 리뷰로 조금 더 학습시켜 쓴다.
- **discriminator (95M)**: 후보를 끼운 문장에서 각 글자가 '바뀐 글자'로 보이는 정도 (ELECTRA 의 원래 학습 과제).

고정·후보는 09 의 캐시 그대로(증강 없는 01 + v2 태거)이므로 LLM 은 전혀 관여하지 않는다.

로컬 확인(검증 앞 100문장, 미세조정 없는 generator): 5-gram 만 0.9862 → 5-gram + generator 0.9903. 100문장뿐이라 참고용이다.

**전제:** 12 노트북 완료 (`outputs/charlm_best.pt`, `best_anchor_agree_v2_k20_b64_uo_12b.json`; test 용 `charlm_full.pt` 는 30 epoch 까지 끝난 것).
**GPU 런타임 (L4 권장), 첫 셀부터.**

## 0) 준비

In [ ]:
import os
os.chdir('/content')
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DRIVE = '/content/drive/MyDrive/데이콘/난독화'
!mkdir -p /content/work && rsync -a --exclude '.playwright-mcp' --exclude '__pycache__' --exclude 'open.zip' --exclude 'v2_cache_*.pkl' --exclude 'models' "$DRIVE/" /content/work/
%cd /content/work
!ls outputs/charlm_best.pt outputs/charlm_full.pt outputs/best_anchor_agree_v2_k20_b64_uo_12b.json outputs/anchor_items_val_agree_v2_k20_b64_uo.pkl outputs/anchor_items_test_agree_v2_k20_b64_uo.pkl 2>&1; nproc; nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
for f in ('charlm_best', 'charlm_full'):
    if os.path.exists(f'outputs/{f}.pt'): print(f, 'epoch', torch.load(f'outputs/{f}.pt', map_location='cpu')['epoch'])
def save_back():
    if not os.path.exists(DRIVE):
        drive.mount('/content/drive', force_remount=True)
    !rsync -a --update --exclude 'v2_cache_*.pkl' /content/work/outputs/ "$DRIVE/outputs/"
    print('saved outputs -> drive')

## 1) generator 미세조정 (train 90%, 5 epoch)
첫 줄 '미세조정 전' 이 사전학습 그대로의 성능이다 (가린 글자 정확도). epoch 마다 검증 손실이 내려가면 리뷰 말투에 적응하는 것이고, 가장 낮은 시점을 `outputs/kochar_gen_split/` 에 저장한다 (`*`). 기록: `outputs/kochar_ft_runs_split.csv`.

In [ ]:
!python -u electra13.py --mode ft --epochs 5
save_back()

## 2) 검증: 가중치 탐색
1단계: w_mlm (generator) × w_rtd (discriminator). 글자 언어모델·5-gram 가중치는 12b 의 최고값으로 고정. **첫 줄 (0, 0) 은 12b 재현이므로 0.9871 이 나와야 정상.**
2단계: 최고 조합에서 12b 항을 하나씩 빼 본다 (사전학습 모델만으로 충분한지).
결과: `outputs/val_pred_anchor_agree_v2_k20_b64_uo_13.csv`, `best_anchor_..._13.json`, `anchor_runs.csv` (name = 13).

In [ ]:
!python -u anchor_electra13.py --mode val --unanchor_oov --cand_beam 64 --nbest 20 --mlm_weights 0,0.25,0.5,1.0 --rtd_weights 0,0.1,0.25,0.5
save_back()

## 3) (선택) 미세조정 효과 확인: 사전학습 그대로의 generator
2 의 `w_rtd 0` 줄들과 비교하면 1 의 미세조정이 얼마나 기여했는지 알 수 있다. 꼬리표 `13z`.

In [ ]:
!python -u anchor_electra13.py --mode val --unanchor_oov --cand_beam 64 --nbest 20 --mlm monologg/kocharelectra-base-generator --mlm_weights 0.25,0.5,1.0 --rtd_weights 0 --name 13z
save_back()

## 4) 결과 비교
최고 가중치가 탐색 범위 끝에 걸렸는지 확인.

In [ ]:
import json
import pandas as pd
r = pd.read_csv('outputs/kochar_ft_runs_split.csv')
BEST_EP = int(r.loc[r.val_loss.idxmin(), 'epoch'])
print('generator 미세조정: 최적 epoch', BEST_EP, '| 검증 손실', r.val_loss.min(), '| 정확도', r.val_acc.max())
a = pd.read_csv('outputs/anchor_runs.csv'); a['name'] = a['name'].astype(str)
g = a[a.name.str.startswith('13')]
print(g.sort_values('f1', ascending=False)[['name', 'w', 'w_kn', 'w_mlm', 'w_rtd', 'f1', 'sec']].head(12))
B = json.load(open('outputs/best_anchor_agree_v2_k20_b64_uo_13.json', encoding='utf-8'))
v13 = pd.read_csv('outputs/val_pred_anchor_agree_v2_k20_b64_uo_13.csv').set_index('ID')
for nm, f in (('01', 'outputs/val_pred.csv'), ('12b', 'outputs/val_pred_anchor_agree_v2_k20_b64_uo_12b.csv'), ('09 (Qwen)', 'outputs/val_pred_anchor_agree_v2_k20_b64_uo_09abc.csv')):
    v = pd.read_csv(f).set_index('ID'); d = v13.f1 - v.f1.loc[v13.index]
    print(f'13 {v13.f1.mean():.4f} vs {nm} {v.f1.mean():.4f} | 좋아진 행 {(d>0).sum()} / 나빠진 행 {(d<0).sum()}')

## 5) test 예측
generator 가중치가 0 보다 크면 train 전체로 다시 미세조정(4 에서 구한 최적 epoch 수)한 뒤 적용한다.
글자 언어모델은 12 노트북이 만든 `outputs/charlm_full.pt` 를 쓴다 (0번 셀 출력에서 epoch 30 인지 확인).
결과: `outputs/submission_anchor_agree_v2_k20_b64_uo_13.csv`

In [ ]:
if B['w_mlm'] > 0:
    !python -u electra13.py --mode ft --full --epochs $BEST_EP
!python -u anchor_electra13.py --mode test --unanchor_oov --cand_beam 64 --nbest 20
save_back()

In [ ]:
!cp "$DRIVE/outputs/charlm_full.pt" outputs/
print('charlm_full epoch', torch.load('outputs/charlm_full.pt', map_location='cpu')['epoch'])
!python -u anchor_electra13.py --mode val --unanchor_oov --cand_beam 64 --nbest 20 --mlm_weights 0 --rtd_weights 0.5,0.75,1.0,1.5,2.0,3.0 --name 13b
!python -u anchor_electra13.py --mode test --unanchor_oov --cand_beam 64 --nbest 20 --name 13b
save_back()


### (선택) 런타임 해제

In [ ]:
from google.colab import runtime; runtime.unassign()